# Notebook 05 — LULC Spatial Analysis
**Part B — Spatial analysis**

## What this notebook does
1. Aggregates 12-day cumulative bioaerosol concentrations per station  
2. Computes **Spearman rank correlations** between 5 LULC variables and 4 bioaerosol targets  
3. Reports **exact p-values** for all 20 pairs  
4. Produces:
   - Spearman heatmap with p-values (Fig 6 in paper)  
   - Scatter plots with regression lines per station (Fig 6 supplement)

## Why Spearman, not Pearson?
Spearman rank correlation makes no assumption about linearity.  
With n=6 stations and non-normally distributed LULC percentages,  
rank correlation is more appropriate than Pearson.

## Why only n=6?
LULC is a **static** property of each station (it does not change day to day).  
Each station contributes exactly one data point to the spatial analysis.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COLAB SETUP — run this cell first if using Google Colab
# ═══════════════════════════════════════════════════════════════
import os, sys

# Option A: Clone the GitHub repo directly in Colab (recommended)
# !git clone https://github.com/Filza-coder/geoai-bioaerosol-prediction.git
# os.chdir('geoai-bioaerosol-prediction')

# Option B: Mount Google Drive and navigate to your folder
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/geoai-bioaerosol-prediction')

# Install dependencies
# !pip install openpyxl geopandas shapely pyproj scikit-learn shap seaborn -q

print('Current directory:', os.getcwd())
print('Python:', sys.version[:10])

In [ ]:
# ── Install (uncomment on Colab) ──────────────────────────────────
# !pip install matplotlib seaborn scipy -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

df      = pd.read_csv('data/df_analysis.csv')
df_lulc = pd.read_csv('data/df_lulc.csv')

for col in ['Aspergillus_conc', 'Alternaria_conc']:
    df[col] = df[col].fillna(df[col].median())

TARGETS = {
    'pollen_conc':      ('Pollen',       '#378ADD'),
    'fungus_conc':      ('Total Fungus', '#D85A30'),
    'Aspergillus_conc': ('Aspergillus',  '#1D9E75'),
    'Alternaria_conc':  ('Alternaria',   '#BA7517'),
}
LULC_VARS = ['veg_pct', 'grass_field_pct', 'barren_pct', 'building_pct', 'tree_count']
LULC_LABS = ['Natural veg (%)', 'Grass fields (%)', 'Barren (%)', 'Buildings (%)', 'Tree count']

print(f'Main dataset: {df.shape}')
print(f'LULC dataset: {df_lulc.shape}')
print(f'\nLULC features per station:')
print(df_lulc.to_string(index=False))

## Step 1 — Aggregate to station level
Sum all 12-day concentrations per station.  
This gives one value per station for each bioaerosol target.

In [ ]:
# Sum 12-day concentrations per station
agg = df.groupby('station')[[t for t in TARGETS]].sum().reset_index()

# Merge with LULC features
spatial = pd.merge(df_lulc, agg, on='station')

print('Station-level data (LULC + cumulative 12-day concentrations):')
print(spatial[['station'] + LULC_VARS + list(TARGETS.keys())].to_string(index=False))

In [ ]:
# ── Compute Spearman correlations + p-values for all 20 pairs ─────
print('Spearman rank correlations (ρ) with p-values — all 20 pairs')
print(f'n = 6 stations | Significance threshold: |ρ| > 0.811 at α = 0.05')
print()
print(f'  {"LULC variable":20s}  {"Target":22s}  ρ       p        Sig')
print('  ' + '-'*72)

all_results = []
for lv, ll in zip(LULC_VARS, LULC_LABS):
    for tcol, (tlabel, col) in TARGETS.items():
        r, p = stats.spearmanr(spatial[lv], spatial[tcol])
        sig  = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        flag = ' ← SIGNIFICANT' if p < 0.05 else ''
        print(f'  {ll:20s}  {tlabel:22s}  {r:+.3f}   {p:.4f}   {sig}{flag}')
        all_results.append({'LULC': ll, 'Target': tlabel, 'rho': r, 'p': p, 'sig': sig})
    print()

## Figure 6 — Spearman heatmap with p-values
Each cell shows ρ (correlation) and significance star.  
Grey = non-significant. Only 3 of 20 pairs are significant at p < 0.05 —  
this is expected and honest at n = 6.

In [ ]:
# Build correlation and p-value matrices
corr_mat = pd.DataFrame(index=LULC_LABS, columns=[v[0] for v in TARGETS.values()], dtype=float)
pval_mat = pd.DataFrame(index=LULC_LABS, columns=[v[0] for v in TARGETS.values()], dtype=float)
anno_mat = pd.DataFrame(index=LULC_LABS, columns=[v[0] for v in TARGETS.values()], dtype=object)

for ll, lv in zip(LULC_LABS, LULC_VARS):
    for tcol, (tlabel, _) in TARGETS.items():
        r, p = stats.spearmanr(spatial[lv], spatial[tcol])
        corr_mat.loc[ll, tlabel] = r
        pval_mat.loc[ll, tlabel] = p
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
        # Cell annotation: ρ value + significance
        anno_mat.loc[ll, tlabel] = f'{r:.2f}\n({sig})'

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(corr_mat.astype(float), annot=anno_mat, fmt='',
            cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 10},
            ax=ax,
            cbar_kws={'label': 'Spearman ρ', 'shrink': 0.85})

ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right', fontsize=10)
ax.set_yticklabels(LULC_LABS, rotation=0, fontsize=9)
ax.set_title('Spearman rank correlations: LULC variables vs cumulative bioaerosol concentrations\n'
             'n = 6 stations | Values show ρ (significance) | * p<0.05  ** p<0.01  ns = not significant\n'
             'Only 3 of 20 pairs are significant — consistent with low statistical power at n=6',
             fontsize=10, pad=10)
plt.tight_layout()
plt.savefig('fig_spearman_lulc.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_spearman_lulc.png')

## Scatter plots for the 3 significant pairs
Natural vegetation vs Aspergillus (ρ=0.829, p=0.042)  
Natural vegetation vs Alternaria (ρ=0.829, p=0.042)  
Barren land vs Total fungus (ρ=−0.880, p=0.021)

Each point is one station, labelled S1–S6.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

sig_pairs = [
    ('veg_pct',    'Natural veg (%)',  'Aspergillus_conc', 'Aspergillus (grain/m³)', '#1D9E75'),
    ('veg_pct',    'Natural veg (%)',  'Alternaria_conc',  'Alternaria (grain/m³)',  '#BA7517'),
    ('barren_pct', 'Barren (%)',       'fungus_conc',      'Total Fungus (grain/m³)','#D85A30'),
]

for i, (lv, ll, tv, tl, col) in enumerate(sig_pairs):
    ax = axes[i]
    x  = spatial[lv].values
    y  = spatial[tv].values
    r, p = stats.spearmanr(x, y)

    # Points
    ax.scatter(x, y, color=col, s=90, zorder=3, edgecolors='white', linewidth=0.5)

    # Label each station
    for xi, yi, si in zip(x, y, spatial['station']):
        ax.annotate(f'S{si}', (xi, yi), fontsize=9, color='#333',
                    xytext=(5, 4), textcoords='offset points')

    # Linear trend line
    z = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 50)
    ax.plot(xs, np.poly1d(z)(xs), '--', color=col, alpha=0.65, lw=1.8)

    ax.set_xlabel(ll, fontsize=10)
    ax.set_ylabel(tl, fontsize=10)
    ax.set_title(f'ρ = {r:.3f}  p = {p:.3f} *', fontsize=10, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Three statistically significant LULC–bioaerosol associations (p < 0.05, n=6)\n'
             'Dashed line = linear trend | S = station number',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('fig_lulc_significant_scatter.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: fig_lulc_significant_scatter.png')

In [ ]:
# ── Summary table ──────────────────────────────────────────────────
print('Summary: All 20 LULC–bioaerosol Spearman pairs')
print(f'{"LULC":20s}  {"Target":22s}  {"ρ":>6}  {"p":>7}  Sig')
print('-'*70)
for res in all_results:
    flag = ' ◄' if res['sig'] != 'ns' else ''
    print(f'{res["LULC"]:20s}  {res["Target"]:22s}  '
          f'{res["rho"]:>+6.3f}  {res["p"]:>7.4f}  {res["sig"]}{flag}')